In [1]:
## Load packages
library(clusterProfiler)
library(org.Hs.eg.db)
library(enrichplot)
library(ggplot2)
library(gridExtra)
library(dplyr)
library(readr)
library(tidyr)
library(patchwork)



clusterProfiler v4.10.0  For help: https://yulab-smu.top/biomedical-knowledge-mining-book/

If you use clusterProfiler in published research, please cite:
T Wu, E Hu, S Xu, M Chen, P Guo, Z Dai, T Feng, L Zhou, W Tang, L Zhan, X Fu, S Liu, X Bo, and G Yu. clusterProfiler 4.0: A universal enrichment tool for interpreting omics data. The Innovation. 2021, 2(3):100141


Attaching package: ‘clusterProfiler’


The following object is masked from ‘package:stats’:

    filter


Loading required package: AnnotationDbi

Loading required package: stats4

Loading required package: BiocGenerics


Attaching package: ‘BiocGenerics’


The following objects are masked from ‘package:stats’:

    IQR, mad, sd, var, xtabs


The following objects are masked from ‘package:base’:

    anyDuplicated, aperm, append, as.data.frame, basename, cbind,
    colnames, dirname, do.call, duplicated, eval, evalq, Filter, Find,
    get, grep, grepl, intersect, is.unsorted, lapply, Map, mapply,
    match, mget, order, 

In [31]:
## Load DMGs and DEGs
deg <- read_csv("tmp_dataset/DEG/DEG_subset_smc_Pericyte_group_comparison.csv")
dmg <- read_tsv("tmp_dataset/DMG/DMG_sub_smc_Pericyte_group_comparison_promoter_imputed_wilcox.tsv")
             
deg <- deg %>%
  ## define DEG as p-value < 0.05, log2FC > 1 and transcript > 10% cells in reference or test population
  mutate(DEG = ifelse(pvals_adj < .05 & abs(logfoldchanges) > 1 & (pct_nz_group > 0.1 | pct_nz_reference > 0.1), TRUE, FALSE)) %>%
  dplyr::rename(gene = names,cluster = comparison) 

dmg <- dmg %>%
  mutate(comparison = paste0(target,"_vs_",reference)) %>%
  group_by(comparison) %>%
  mutate(qval = p.adjust(pval, method = "BH")) %>%    
  ungroup() %>%
  ## define DMG as p-value < 0.05, delta > 0.15 and methylation coverage deteted in > 10% cells in reference and test population
  mutate(DMG = ifelse(qval < .05 & abs(delta) > .15 & (pct_valid_target > 0.1 & pct_valid_ref > 0.1), TRUE, FALSE)) %>% 
  dplyr::rename(gene = var, cluster = comparison)

combine.deg.dmg <- deg %>%
  inner_join(dmg, by = c("cluster","gene"))



Rows: 115818 Columns: 10
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (4): names, target, reference, comparison
dbl (6): scores, logfoldchanges, pvals, pvals_adj, pct_nz_group, pct_nz_refe...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 112902 Columns: 13
── Column specification ────────────────────────────────────────────────────────
Delimiter: "\t"
chr (4): var, bin_chunk, target, reference
dbl (8): stat, pval, qval, mean_target, mean_ref, delta, pct_valid_target, p...
lgl (1): DMG

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


In [33]:
ego_hyper <- list()
ego_hypo <- list()
safe_barplot <- function(ego_obj, title_text) {

  if (is.null(ego_obj) || nrow(as.data.frame(ego_obj)) == 0) {
    return(
      ggplot() +
        theme_void() +
        ggtitle(paste0(title_text, "\n(no enrichment)"))
    )
  }

  barplot(ego_obj, showCategory = 10) +
    ggtitle(title_text)
}
for(items in c("G2_Pericyte_vs_G1_Pericyte","G3_Pericyte_vs_G1_Pericyte","G3_Pericyte_vs_G2_Pericyte")) {
dmg_df <- combine.deg.dmg %>%
  filter(cluster == items & DEG) 
dmg_hyper <- dmg_df %>% filter(logfoldchanges > 0)
dmg_hypo  <- dmg_df %>% filter(logfoldchanges < 0)

safe_bitr <- function(expr) {
  tryCatch(expr, error = function(e) NULL)
}
           
convert_to_entrez <- function(gene_vec) {

  gene_vec <- unique(na.omit(gene_vec))
  if (length(gene_vec) == 0) return(character(0))

  gene_vec <- sub("\\..*", "", gene_vec)

  is_ensembl <- grepl("^ENSG", gene_vec)

  genes_symbol  <- gene_vec[!is_ensembl]
  genes_ensembl <- gene_vec[is_ensembl]

  df_symbol <- NULL
  df_ensembl <- NULL

  if (length(genes_symbol) > 0) {
    df_symbol <- safe_bitr(
      bitr(genes_symbol,
           fromType = "SYMBOL",
           toType = "ENTREZID",
           OrgDb = org.Hs.eg.db)
    )
  }

  if (length(genes_ensembl) > 0) {
    df_ensembl <- safe_bitr(
      bitr(genes_ensembl,
           fromType = "ENSEMBL",
           toType = "ENTREZID",
           OrgDb = org.Hs.eg.db)
    )
  }

  df_all <- bind_rows(df_symbol, df_ensembl)

  if (is.null(df_all) || nrow(df_all) == 0) {
    return(character(0))   
  }

  df_all %>%
    distinct(ENTREZID) %>%
    pull(ENTREZID)
}

entrez_hyper <- convert_to_entrez(dmg_hyper$gene)
entrez_hypo  <- convert_to_entrez(dmg_hypo$gene)

ego_hyper[items] <- enrichGO(gene          = entrez_hyper,
                      OrgDb         = org.Hs.eg.db,
                      ont           = "BP",
                      pAdjustMethod = "BH",
                      pvalueCutoff  = 0.05,
                      readable      = TRUE)

ego_hypo[items] <- enrichGO(gene          = entrez_hypo,
                     OrgDb         = org.Hs.eg.db,
                     ont           = "BP",
                     pAdjustMethod = "BH",
                     pvalueCutoff  = 0.05,
                     readable      = TRUE)

}
p1 <- safe_barplot(ego_hyper[[1]],"G2 vs G1 Up")
p2 <- safe_barplot(ego_hypo[[1]],"G2 vs G1 Down")

p3 <- safe_barplot(ego_hyper[[2]],"G3 vs G1 Up")
p4 <- safe_barplot(ego_hypo[[2]],"G3 vs G1 Down")

p5 <- safe_barplot(ego_hyper[[3]],"G3 vs G2 Up")
p6 <- safe_barplot(ego_hypo[[3]],"G3 vs G2 Down")

pdf("figures/GO_enrichment_DEGs_Pericyte_groupcomp.pdf", width = 16, height = 22)
(p1 | p2 ) / (p3 | p4 )/ ( p5 | p6)
dev.off()

'select()' returned 1:1 mapping between keys and columns

'select()' returned 1:1 mapping between keys and columns

Warning message in `[<-`(`*tmp*`, items, value = enrichGO(gene = entrez_hyper, OrgDb = org.Hs.eg.db, :
“implicit list embedding of S4 objects is deprecated”
Warning message in `[<-`(`*tmp*`, items, value = enrichGO(gene = entrez_hypo, OrgDb = org.Hs.eg.db, :
“implicit list embedding of S4 objects is deprecated”
'select()' returned 1:1 mapping between keys and columns

'select()' returned 1:1 mapping between keys and columns

'select()' returned 1:1 mapping between keys and columns

Warning message in bitr(genes_ensembl, fromType = "ENSEMBL", toType = "ENTREZID", :
“66.67% of input gene IDs are fail to map...”
Warning message in `[<-`(`*tmp*`, items, value = enrichGO(gene = entrez_hyper, OrgDb = org.Hs.eg.db, :
“implicit list embedding of S4 objects is deprecated”
Warning message in `[<-`(`*tmp*`, items, value = enrichGO(gene = entrez_hypo, OrgDb = org.Hs.eg.db, :
“implici

pdf 
  2

In [34]:
## save significant GO terms and genes as table
make_go_table <- function(ego_obj, group_name, n = 10) {

  if (is.null(ego_obj) || nrow(as.data.frame(ego_obj)) == 0) {
    return(data.frame(
      Group = group_name,
      GO_term = NA,
      Description = "No enrichment",
      Gene = NA
    ))
  }
  df <- as.data.frame(ego_obj)
  top_df <- df %>%
    arrange(p.adjust) %>%
    head(n)
  out <- top_df %>%
    mutate(
      Group = group_name,
      Gene = geneID   
    ) 
  return(out)
}

go_table <- bind_rows(
  make_go_table(ego_hyper[[1]], "G2_vs_G1_Up"),
  make_go_table(ego_hypo[[1]],  "G2_vs_G1_Down"),

  make_go_table(ego_hyper[[2]], "G3_vs_G1_Up"),
  make_go_table(ego_hypo[[2]],  "G3_vs_G1_Down"),

  make_go_table(ego_hyper[[3]], "G3_vs_G2_Up"),
  make_go_table(ego_hypo[[3]],  "G3_vs_G2_Down")
)

dmg_grouped <- combine.deg.dmg %>%
  filter(DMG) %>%
  mutate(group = case_when(
    cluster == "G2_Pericyte_vs_G1_Pericyte" ~ "G2_vs_G1",
    cluster == "G3_Pericyte_vs_G1_Pericyte" ~ "G3_vs_G1",
    cluster == "G3_Pericyte_vs_G2_Pericyte" ~ "G3_vs_G2",
    TRUE ~ cluster
  )) %>%
  mutate(direction = case_when(
    delta > 0 ~ "Down",
    delta < 0 ~ "Up",
    TRUE ~ NA_character_
  ))

dmg_map <- dmg_grouped %>%
  mutate(Group = paste0(group, "_", direction)) %>%
  group_by(Group) %>%
  summarise(DMG_genes = list(unique(gene)), .groups = "drop")

go_clean <- go_table %>%
  filter(!is.na(Gene)) %>%
  mutate(Gene = as.character(Gene)) %>%
  separate_rows(Gene, sep = "/") %>%
  mutate(Gene = trimws(Gene))

go_overlap <- go_clean %>%
  left_join(dmg_map, by = "Group") %>%
  mutate(
    is_DMG = ifelse(is.na(DMG_genes), FALSE, Gene %in% unlist(DMG_genes))
  )
go_overlap_summary <- go_overlap %>%
  group_by(Group, ID, Description) %>%
  summarise(
    n_genes_total = n(),
    n_DMG = sum(is_DMG),
    overlap_ratio = n_DMG / n_genes_total,
    DMG_genes = paste(unique(Gene[is_DMG]), collapse = "/"),
    all_genes = paste(unique(Gene), collapse = "/"),
    .groups = "drop"
  ) %>%
  arrange(desc(overlap_ratio))

go_table_new <- go_table %>%
  left_join(
    dplyr::select(go_overlap_summary, Group, ID, n_DMG, overlap_ratio, DMG_genes),
    by = c("Group", "ID")
  )
write_tsv(go_table_new, "tmp_dataset/GO_top10_DEG_pathways_Pericyte.csv")